In [ ]:
# Run this cell first to install required packages
!pip install rouge-score
!pip install bert-score
!pip install nltk
!pip install transformers==4.51.0  # Use stable version
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install accelerate
!pip install tqdm  # Progress bar


# Download NLTK data
import nltk
nltk.download('punkt')

In [ ]:
import os
import pandas as pd
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, pipeline
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu
from bert_score import score
import json
import warnings
import gc
from tqdm import tqdm
import re
import unicodedata
warnings.filterwarnings("ignore")

# Set environment variable for better CUDA error debugging
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

def normalize_urdu_text(text):
    """
    Normalize Urdu text for better comparison
    """
    if not text:
        return ""
    
    # Convert to string and strip
    text = str(text).strip()
    
    # Normalize Unicode (important for Urdu)
    text = unicodedata.normalize('NFKC', text)
    
    # Remove extra whitespaces
    text = re.sub(r'\s+', ' ', text)
    
    # Remove common punctuation but keep Urdu punctuation
    text = re.sub(r'[۔،؍؎؏ؘؙؚؐؑؒؓؔؕؖؗ؛؜]', '', text)
    
    return text.strip()

def calculate_urdu_rouge(reference, generated):
    """
    Calculate ROUGE scores with proper Urdu text normalization
    """
    # Normalize both texts
    ref_normalized = normalize_urdu_text(reference)
    gen_normalized = normalize_urdu_text(generated)
    
    # Tokenize by spaces (simple but effective for Urdu)
    ref_tokens = ref_normalized.split()
    gen_tokens = gen_normalized.split()
    
    if not ref_tokens or not gen_tokens:
        return {'rouge1': 0.0, 'rouge2': 0.0, 'rougeL': 0.0}
    
    # ROUGE-1: unigram overlap
    ref_unigrams = set(ref_tokens)
    gen_unigrams = set(gen_tokens)
    
    if len(gen_unigrams) == 0:
        rouge1 = 0.0
    else:
        rouge1_overlap = len(ref_unigrams.intersection(gen_unigrams))
        rouge1_precision = rouge1_overlap / len(gen_unigrams)
        rouge1_recall = rouge1_overlap / len(ref_unigrams)
        rouge1 = 2 * rouge1_precision * rouge1_recall / (rouge1_precision + rouge1_recall) if (rouge1_precision + rouge1_recall) > 0 else 0.0
    
    # ROUGE-2: bigram overlap
    ref_bigrams = set(zip(ref_tokens, ref_tokens[1:])) if len(ref_tokens) > 1 else set()
    gen_bigrams = set(zip(gen_tokens, gen_tokens[1:])) if len(gen_tokens) > 1 else set()
    
    if len(gen_bigrams) == 0 or len(ref_bigrams) == 0:
        rouge2 = 0.0
    else:
        rouge2_overlap = len(ref_bigrams.intersection(gen_bigrams))
        rouge2_precision = rouge2_overlap / len(gen_bigrams)
        rouge2_recall = rouge2_overlap / len(ref_bigrams)
        rouge2 = 2 * rouge2_precision * rouge2_recall / (rouge2_precision + rouge2_recall) if (rouge2_precision + rouge2_recall) > 0 else 0.0
    
    # ROUGE-L: Longest Common Subsequence
    def lcs_length(X, Y):
        m, n = len(X), len(Y)
        dp = [[0] * (n + 1) for _ in range(m + 1)]
        
        for i in range(1, m + 1):
            for j in range(1, n + 1):
                if X[i-1] == Y[j-1]:
                    dp[i][j] = dp[i-1][j-1] + 1
                else:
                    dp[i][j] = max(dp[i-1][j], dp[i][j-1])
        return dp[m][n]
    
    lcs_len = lcs_length(ref_tokens, gen_tokens)
    if len(gen_tokens) == 0 or len(ref_tokens) == 0:
        rougeL = 0.0
    else:
        rougeL_precision = lcs_len / len(gen_tokens)
        rougeL_recall = lcs_len / len(ref_tokens)
        rougeL = 2 * rougeL_precision * rougeL_recall / (rougeL_precision + rougeL_recall) if (rougeL_precision + rougeL_recall) > 0 else 0.0
    
    return {
        'rouge1': rouge1,
        'rouge2': rouge2,
        'rougeL': rougeL
    }

# Safe GPU cache clearing with error handling
def safe_clear_cache():
    try:
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()
    except RuntimeError as e:
        print(f"Warning: Could not clear CUDA cache: {e}")
        print("Continuing with CPU...")
        return False
    return True

# Try to clear cache, if it fails, we'll use CPU
cuda_available = safe_clear_cache()

# Set device with better error handling
if torch.cuda.is_available() and cuda_available:
    device = "cuda"
    try:
        # Test CUDA functionality
        test_tensor = torch.tensor([1.0]).cuda()
        test_tensor = test_tensor.cpu()
        del test_tensor
        print(f"CUDA available and working. GPU: {torch.cuda.get_device_name()}")
        print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
    except RuntimeError as e:
        print(f"CUDA test failed: {e}")
        print("Falling back to CPU")
        device = "cpu"
        cuda_available = False
else:
    device = "cpu"
    cuda_available = False
    print("Using CPU")

# Load model and tokenizer with better memory management
model_path = "/kaggle/input/finetunedmbartlarge67kwith50news147scripts/pytorch/default/1/finetunedmBartLarge67kWith50news147Scripts"

try:
    print("Loading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    
    print("Loading model...")
    # Load model with single GPU configuration to avoid device mismatch
    if device == "cuda" and cuda_available:
        model = AutoModelForSeq2SeqLM.from_pretrained(
            model_path,
            torch_dtype=torch.float16,
            low_cpu_mem_usage=True,
            device_map={"": 0}  # Force all model parts to GPU 0
        )
    else:
        model = AutoModelForSeq2SeqLM.from_pretrained(
            model_path,
            torch_dtype=torch.float32,
            low_cpu_mem_usage=True
        )
        model = model.to(device)
    
    print(f"Model loaded successfully on {device}")
    
except Exception as e:
    print(f"Error loading model: {e}")
    print("Trying with CPU fallback...")
    device = "cpu"
    cuda_available = False
    model = AutoModelForSeq2SeqLM.from_pretrained(
        model_path,
        torch_dtype=torch.float32,
        low_cpu_mem_usage=True
    ).to(device)

# Initialize original ROUGE scorer for comparison
rouge_scorer_obj = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

# Load dataset
dataset_path = "/kaggle/input/urdu-xlsum/urdu_natural_instructions.json"
with open(dataset_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

# Select first 250 instances
instances = data['Instances'][:250]

# Function to clean generated text
def extract_summary(generated_text):
    prefixes = ["Summary :", "Summary:", "Summary ", "Summary Summary : ", "Summary Summary:"]
    cleaned_text = str(generated_text)
    for prefix in prefixes:
        if cleaned_text.startswith(prefix):
            cleaned_text = cleaned_text[len(prefix):].strip()
    return cleaned_text

# Function to truncate summary to complete sentences within token limit
def truncate_to_complete_sentence(text, max_tokens=35):
    """
    Truncate text to fit within max_tokens while ensuring it ends at a complete sentence.
    """
    # Tokenize the text to count tokens
    tokens = tokenizer.tokenize(text)
    
    # If already within limit, return as is
    if len(tokens) <= max_tokens:
        return text
    
    # Truncate to max_tokens
    truncated_tokens = tokens[:max_tokens]
    truncated_text = tokenizer.convert_tokens_to_string(truncated_tokens)
    
    # Find the last complete sentence (ending with full stop or Urdu full stop)
    # Split by periods and Urdu full stops
    for delimiter in ['۔', '.']:
        if delimiter in truncated_text:
            sentences = truncated_text.split(delimiter)
            if len(sentences) > 1:
                # Join all complete sentences (excluding the last potentially incomplete one)
                complete_text = delimiter.join(sentences[:-1]).strip()
                if complete_text:  # Make sure we have some content
                    return complete_text + delimiter
    
    # If no complete sentence found, return the truncated text as is (fallback)
    return truncated_text

# Lists to store results
results = []

# Process each instance with memory management
progress_bar = tqdm(enumerate(instances), total=len(instances), desc="Processing instances")

for idx, instance in progress_bar:
    try:
        input_text = instance['input']
        reference_summary = instance['output'][0] if isinstance(instance['output'], list) else instance['output']

        # Generate summary with memory optimization
        input_text_formatted = f"Text: {input_text}"
        
        # Handle long sequences by truncating if necessary
        max_input_length = 1020  # Leave some room for the "Text: " prefix
        input_ids = tokenizer(input_text_formatted, return_tensors="pt", truncation=True, max_length=max_input_length)
        
        # Move input to the same device as model
        if device == "cuda" and cuda_available:
            input_ids = {k: v.to("cuda:0") for k, v in input_ids.items()}
        
        input_size = input_ids["input_ids"].shape[1]
        
        # Generate with controlled parameters - set max_new_tokens to 50
        with torch.no_grad():
            generated = model.generate(
                input_ids["input_ids"],
                attention_mask=input_ids["attention_mask"],
                max_new_tokens=35,  # Generate maximum 50 new tokens
                min_length=20,
                do_sample=False,
                num_beams=1,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id
            )
        
        # Decode generated text
        generated_text = tokenizer.decode(generated[0], skip_special_tokens=True)
        generated_summary = extract_summary(generated_text)
        
        # Truncate to complete sentence within 50 tokens
        final_summary = truncate_to_complete_sentence(generated_summary, max_tokens=50)

        # Calculate ROUGE scores using both methods
        # Original ROUGE (likely broken for Urdu)
        try:
            rouge_scores_original = rouge_scorer_obj.score(reference_summary, final_summary)
            rouge1_orig = rouge_scores_original['rouge1'].fmeasure
            rouge2_orig = rouge_scores_original['rouge2'].fmeasure
            rougeL_orig = rouge_scores_original['rougeL'].fmeasure
        except:
            rouge1_orig = rouge2_orig = rougeL_orig = 0.0
        
        # Custom Urdu ROUGE
        urdu_rouge_scores = calculate_urdu_rouge(reference_summary, final_summary)
        rouge1 = urdu_rouge_scores['rouge1']
        rouge2 = urdu_rouge_scores['rouge2']
        rougeL = urdu_rouge_scores['rougeL']

        # Calculate BLEU score with error handling
        try:
            reference_tokens = tokenizer.tokenize(reference_summary)
            generated_tokens = tokenizer.tokenize(final_summary)
            
            if len(generated_tokens) > 0 and len(reference_tokens) > 0:
                bleu_score = sentence_bleu([reference_tokens], generated_tokens)
            else:
                bleu_score = 0.0
        except:
            bleu_score = 0.0

        # Calculate BERTScore with error handling
        try:
            P, R, F1 = score([final_summary], [reference_summary], lang="ur", model_type="xlm-roberta-base")
            bert_f1 = F1.item()
        except:
            bert_f1 = 0.0

        # Count tokens in final summary for verification
        final_token_count = len(tokenizer.tokenize(final_summary))

        # Store results
        results.append({
            'id': instance['id'],
            'input': input_text_formatted,
            'reference_summary': reference_summary,
            'generated_summary': final_summary,
            'token_count': final_token_count,
            'rouge1': rouge1,
            'rouge2': rouge2,
            'rougeL': rougeL,
            'rouge1_original': rouge1_orig,
            'rouge2_original': rouge2_orig,
            'rougeL_original': rougeL_orig,
            'bleu': bleu_score,
            'bertscore_f1': bert_f1
        })

        # Clear cache periodically
        if (idx + 1) % 10 == 0:
            safe_clear_cache()

        # Update progress bar description with current metrics
        if len(results) > 0:
            current_rouge1 = sum(r['rouge1'] for r in results) / len(results)
            current_rouge1_orig = sum(r['rouge1_original'] for r in results) / len(results)
            avg_tokens = sum(r['token_count'] for r in results) / len(results)
            progress_bar.set_postfix({
                'Urdu_R1': f'{current_rouge1:.3f}', 
                'Orig_R1': f'{current_rouge1_orig:.3f}',
                'Avg_Tokens': f'{avg_tokens:.1f}', 
                'Processed': len(results)
            })
        
    except Exception as e:
        progress_bar.write(f"Error processing instance {idx + 1}: {e}")
        continue

progress_bar.close()

# Create DataFrame
df_results = pd.DataFrame(results)

# Compute average scores
if len(df_results) > 0:
    # Urdu-aware ROUGE scores
    avg_rouge1 = df_results['rouge1'].mean()
    avg_rouge2 = df_results['rouge2'].mean()
    avg_rougeL = df_results['rougeL'].mean()
    
    # Original ROUGE scores (for comparison)
    avg_rouge1_orig = df_results['rouge1_original'].mean()
    avg_rouge2_orig = df_results['rouge2_original'].mean()
    avg_rougeL_orig = df_results['rougeL_original'].mean()
    
    avg_bleu = df_results['bleu'].mean()
    avg_bertscore = df_results['bertscore_f1'].mean()
    avg_token_count = df_results['token_count'].mean()

    # Print average scores
    print(f"\nProcessed {len(df_results)} instances successfully")
    print(f"\nAverage token count: {avg_token_count:.2f}")
    
    print("\n" + "="*60)
    print("URDU-AWARE ROUGE SCORES:")
    print("="*60)
    print(f"ROUGE-1: {avg_rouge1:.4f}")
    print(f"ROUGE-2: {avg_rouge2:.4f}")
    print(f"ROUGE-L: {avg_rougeL:.4f}")
    
    print("\n" + "="*60)
    print("ORIGINAL ROUGE SCORES (for comparison):")
    print("="*60)
    print(f"ROUGE-1: {avg_rouge1_orig:.4f}")
    print(f"ROUGE-2: {avg_rouge2_orig:.4f}")
    print(f"ROUGE-L: {avg_rougeL_orig:.4f}")
    
    print("\n" + "="*60)
    print("OTHER METRICS:")
    print("="*60)
    print(f"BLEU: {avg_bleu:.4f}")
    print(f"BERTScore F1: {avg_bertscore:.4f}")

    # Print token count distribution
    print(f"\nToken count statistics:")
    print(f"Min tokens: {df_results['token_count'].min()}")
    print(f"Max tokens: {df_results['token_count'].max()}")
    print(f"Summaries with exactly 50 tokens: {(df_results['token_count'] == 50).sum()}")
    print(f"Summaries with <50 tokens: {(df_results['token_count'] < 50).sum()}")

    # Create output directory if it doesn't exist
    os.makedirs("/kaggle/working/output", exist_ok=True)

    # Save results to CSV
    output_csv_path = "/kaggle/working/output/mbart_summaries_urdu_rouge_fixed.csv"
    df_results.to_csv(output_csv_path, index=False, encoding='utf-8')
    print(f"\nResults saved to {output_csv_path}")

    # Zip the output for download
    import zipfile
    from IPython.display import FileLink
    
    zip_name = "/kaggle/working/mbart_summaries_urdu_rouge_fixed.zip"
    with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zipf:
        zipf.write(output_csv_path, os.path.basename(output_csv_path))
    
    print(f"Results zipped to {zip_name}")
    FileLink(zip_name)
else:
    print("No instances were processed successfully!")

# Final cleanup
safe_clear_cache()